# IndoBERT Sentiment Fine-Tuning on Google Colab


## Research Context
This notebook runs IndoBERT fine-tuning for the SentiRank sentiment classification workflow. IndoBERT is used only for sentiment classification, with three labels: `Negative`, `Neutral`, and `Positive`. Aspect classification remains a separate SVM workflow and is not trained here.

## Environment Setup
Run this notebook in Google Colab with GPU runtime enabled. The package install cell is intended for Colab only.

In [ ]:
!pip -q install transformers datasets evaluate accelerate scikit-learn pandas matplotlib


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU runtime is required for controlled IndoBERT fine-tuning.")


## Google Drive Mount
Set `DRIVE_PROJECT_DIR` to the project folder in Google Drive after uploading the IndoBERT dataset bundle. Do not hardcode a personal path throughout the notebook; update the variable once here.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/SentiRank")
DATASET_DIR = DRIVE_PROJECT_DIR / "datasets" / "processed" / "indobert"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs" / "indobert"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "saved_model"
FIGURES_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Dataset Paths
Expected files in `DATASET_DIR`:
- `train.csv`
- `validation.csv`
- `test.csv`
- `label_mapping.json`

These files come from local Phase 8A output: `datasets/processed/indobert/`.

In [ ]:
TRAIN_FILE = DATASET_DIR / "train.csv"
VALIDATION_FILE = DATASET_DIR / "validation.csv"
TEST_FILE = DATASET_DIR / "test.csv"
LABEL_MAPPING_FILE = DATASET_DIR / "label_mapping.json"

for path in [TRAIN_FILE, VALIDATION_FILE, TEST_FILE, LABEL_MAPPING_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required dataset file: {path}")


## Dataset Loading and Validation
The required columns are `text_indobert` and `label_id`. Label IDs must match `0 = Negative`, `1 = Neutral`, and `2 = Positive`.

In [ ]:
import json
import random

import numpy as np
import pandas as pd

REQUIRED_COLUMNS = {"text_indobert", "label_id"}
EXPECTED_LABEL_MAPPING = {"Negative": 0, "Neutral": 1, "Positive": 2}

def load_split(path: Path, split_name: str) -> pd.DataFrame:
    data = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS - set(data.columns)
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {sorted(missing_columns)}")
    empty_text_count = data["text_indobert"].fillna("").astype(str).str.strip().eq("").sum()
    if empty_text_count:
        raise ValueError(f"{split_name} contains {empty_text_count} empty text rows")
    return data

train_df = load_split(TRAIN_FILE, "train")
validation_df = load_split(VALIDATION_FILE, "validation")
test_df = load_split(TEST_FILE, "test")
label_mapping = json.loads(LABEL_MAPPING_FILE.read_text(encoding="utf-8"))

if label_mapping != EXPECTED_LABEL_MAPPING:
    raise ValueError(f"Unexpected label mapping: {label_mapping}")

id2label = {label_id: label for label, label_id in label_mapping.items()}
label2id = label_mapping

print("Dataset sizes:")
print({"train": len(train_df), "validation": len(validation_df), "test": len(test_df)})

for split_name, data in {"train": train_df, "validation": validation_df, "test": test_df}.items():
    print(f"\n{split_name} label distribution")
    print(data["label_id"].map(id2label).value_counts().sort_index())


## Training Configuration
Default configuration:
- `model_name = indobenchmark/indobert-base-p1`
- `max_length = 128`
- `batch_size = 16`
- `learning_rate = 2e-5`
- `epochs = 3`
- `random_state = 42`
- `metric_for_best_model = f1_macro`

If GPU memory is limited, use `batch_size = 8` and `gradient_accumulation_steps = 2`.

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 128
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 2e-5
EPOCHS = 3
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)


## Tokenization
Use `AutoTokenizer` and tokenize `text_indobert` with truncation to `MAX_LENGTH`.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(data: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(
        data[["text_indobert", "label_id"]].rename(columns={"label_id": "labels"}),
        preserve_index=False,
    )

def tokenize_batch(batch):
    return tokenizer(batch["text_indobert"], truncation=True, max_length=MAX_LENGTH)

train_dataset = to_hf_dataset(train_df).map(tokenize_batch, batched=True)
validation_dataset = to_hf_dataset(validation_df).map(tokenize_batch, batched=True)
test_dataset = to_hf_dataset(test_df).map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## Model Setup
Use `AutoModelForSequenceClassification` with `num_labels = 3`. The model configuration must keep `id2label` and `label2id` aligned with the local label mapping.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)


## Metrics
Compute accuracy, macro precision, macro recall, macro F1, weighted precision, weighted recall, and weighted F1.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
    }


## Training
Use Hugging Face `Trainer`, save the best model, and apply early stopping. This cell starts the actual fine-tuning job.

In [ ]:
import inspect

from transformers import EarlyStoppingCallback, Trainer, TrainingArguments

training_argument_names = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
strategy_key = "eval_strategy" if "eval_strategy" in training_argument_names else "evaluation_strategy"
training_kwargs = {
    "output_dir": str(OUTPUT_DIR / "checkpoints"),
    strategy_key: "epoch",
    "save_strategy": "epoch",
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": EPOCHS,
    "weight_decay": 0.01,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "greater_is_better": True,
    "report_to": [],
    "seed": RANDOM_STATE,
}
if "logging_strategy" in training_argument_names:
    training_kwargs["logging_strategy"] = "epoch"
if "save_total_limit" in training_argument_names:
    training_kwargs["save_total_limit"] = 1

training_args = TrainingArguments(**training_kwargs)

trainer_argument_names = set(inspect.signature(Trainer.__init__).parameters.keys())
trainer_tokenizer_kwargs = (
    {"processing_class": tokenizer}
    if "processing_class" in trainer_argument_names
    else {"tokenizer": tokenizer}
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    **trainer_tokenizer_kwargs,
)

train_result = trainer.train()
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))


## Evaluation
Evaluate on the test set, generate a classification report, create a confusion matrix, and save predictions.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_metrics = trainer.evaluate(test_dataset, metric_key_prefix="test")
prediction_output = trainer.predict(test_dataset)
test_logits = prediction_output.predictions
test_predictions = np.argmax(test_logits, axis=-1)
test_labels = prediction_output.label_ids

classification_report_dict = classification_report(
    test_labels,
    test_predictions,
    labels=[0, 1, 2],
    target_names=[id2label[0], id2label[1], id2label[2]],
    output_dict=True,
    zero_division=0,
)
confusion_matrix_data = confusion_matrix(test_labels, test_predictions, labels=[0, 1, 2])

predictions_df = test_df.copy()
predictions_df["true_label_id"] = test_labels
predictions_df["true_label"] = [id2label[int(label_id)] for label_id in test_labels]
predictions_df["predicted_label_id"] = test_predictions
predictions_df["predicted_label"] = [id2label[int(label_id)] for label_id in test_predictions]
predictions_df.to_csv(OUTPUT_DIR / "indobert_test_predictions.csv", index=False)

test_metrics


## Output Saving
Save the trained model, tokenizer, metrics, predictions, classification report, confusion matrix, and figures to Google Drive.

In [ ]:
import matplotlib.pyplot as plt

metrics_payload = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "random_state": RANDOM_STATE,
    "train_metrics": train_result.metrics,
    "test_metrics": test_metrics,
}

(OUTPUT_DIR / "indobert_training_metrics.json").write_text(
    json.dumps(metrics_payload, indent=2, sort_keys=True), encoding="utf-8"
)
(OUTPUT_DIR / "indobert_classification_report.json").write_text(
    json.dumps(classification_report_dict, indent=2, sort_keys=True), encoding="utf-8"
)
pd.DataFrame(
    confusion_matrix_data,
    index=[id2label[0], id2label[1], id2label[2]],
    columns=[id2label[0], id2label[1], id2label[2]],
).to_csv(OUTPUT_DIR / "indobert_confusion_matrix.csv")

metric_names = ["test_accuracy", "test_precision_macro", "test_recall_macro", "test_f1_macro", "test_f1_weighted"]
metric_values = [float(test_metrics.get(name, 0.0)) for name in metric_names]
plt.figure(figsize=(9, 5))
plt.bar(metric_names, metric_values, color="#2f6f9f")
plt.title("IndoBERT Test Metrics")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "indobert_eval_metrics.png", dpi=160)
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(confusion_matrix_data, cmap="Blues")
plt.title("IndoBERT Confusion Matrix")
plt.xticks([0, 1, 2], [id2label[0], id2label[1], id2label[2]])
plt.yticks([0, 1, 2], [id2label[0], id2label[1], id2label[2]])
plt.xlabel("Predicted")
plt.ylabel("True")
for row_index in range(confusion_matrix_data.shape[0]):
    for column_index in range(confusion_matrix_data.shape[1]):
        plt.text(column_index, row_index, confusion_matrix_data[row_index, column_index], ha="center", va="center")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "indobert_confusion_matrix.png", dpi=160)
plt.show()

loss_points = [(entry.get("epoch"), entry.get("loss")) for entry in trainer.state.log_history if "loss" in entry]
eval_loss_points = [(entry.get("epoch"), entry.get("eval_loss")) for entry in trainer.state.log_history if "eval_loss" in entry]
if loss_points or eval_loss_points:
    plt.figure(figsize=(7, 4))
    if loss_points:
        epochs, losses = zip(*loss_points)
        plt.plot(epochs, losses, marker="o", label="Training loss")
    if eval_loss_points:
        epochs, losses = zip(*eval_loss_points)
        plt.plot(epochs, losses, marker="o", label="Validation loss")
    plt.title("IndoBERT Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "indobert_training_loss.png", dpi=160)
    plt.show()


## Local Repository Sync Note
Only small metrics and figures should be copied back into the repository:
- `datasets/outputs/eda/03_indobert/`
- `docs/figures/03_indobert/`

Do not copy trained model weights, checkpoints, tokenizer files, or large prediction artifacts into Git.

## Interpretation Section
Document final accuracy, macro F1, weighted F1, per-class behavior, and confusion matrix patterns here after training. Pay particular attention to class imbalance and potential difficulty in the `Neutral` class.

## Next Step
Continue to `05_model_evaluation.ipynb` after Colab metrics and figures are copied back. SVM aspect classifier training should be handled later, after aspect label quality control.